# 🔍 Customer Segmentation Analysis
### ABC Debt Relief  |  Data-Driven Portfolio Strategy  |  5 Customer Segments  |  K-Means Clustering

**Company:** ABC Debt Relief
**Objective:** Identify distinct customer segments from ABC Debt Relief's credit card behavioural data
to drive targeted retention, growth, and risk management strategies across the portfolio.
**Dataset:** Proprietary customer snapshot — ABC Debt Relief internal data
**Classification:** Confidential — Internal Use Only

---

## 📋 Table of Contents
1. [Setup & Library Imports](#setup)
2. [Data Pull & Initial Overview](#data)
3. [Missing Value Treatment](#missing)
4. [Exploratory Data Analysis (EDA)](#eda)
5. [Feature Engineering](#features)
6. [Outlier Detection & Removal](#outliers)
7. [Data Preprocessing & Scaling](#preprocessing)
8. [Cluster Selection — Elbow Method (Inertia)](#elbow)
9. [Cluster Selection — Silhouette Score & Validation Metrics](#silhouette)
10. [Model Comparison: K-Means vs Hierarchical vs DBSCAN vs GMM](#comparison)
11. [Final Model — K-Means k=5](#finalmodel)
12. [PCA Cluster Visualisation](#pca)
13. [Segment Profiles & Spider / Radar Charts](#segments)
14. [Portfolio Distribution & ROI Analysis](#portfolio)
15. [Recommendations & 90-Day Implementation Roadmap](#recommendations)

---
<a id='setup'></a>
## 1️⃣ Setup & Library Imports

In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import math

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.cluster       import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture       import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics       import (silhouette_score, davies_bouldin_score,
                                   calinski_harabasz_score)
from sklearn.ensemble      import IsolationForest
from sklearn.neighbors     import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage

# Optional — knee detection
try:
    from kneed import KneeLocator
    KNEED = True
except ImportError:
    KNEED = False
    print("kneed not installed — pip install kneed for automated knee detection")

# ── Colour palette ────────────────────────────────────────────────────────────
PURPLE   = '#667eea'; DARK_PUR = '#764ba2'; LIGHT_BG = '#f0f4ff'
SEG_COLS = {
    'Distressed Revolvers': '#d32f2f',
    'Prime Customers':      '#2ecc71',
    'Engaged Optimizers':   '#ff9800',
    'Low Cap / Disengaged': '#2196f3',
    'High Risk / New':      '#9c27b0',
}
SEG_ORDER = list(SEG_COLS.keys())
ALL_COLS  = list(SEG_COLS.values())

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#fafbff',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'sans-serif',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'figure.dpi':       110,
})

COMPANY = 'ABC Debt Relief'
print(f"✅ All libraries loaded successfully")
print(f"   Analysis prepared for: {COMPANY}")

---
<a id='data'></a>
## 2️⃣ Data Pull & Initial Overview

**Source:** ABC Debt Relief internal customer database.
Credit-card behavioural snapshot for ABC Debt Relief's cardholder portfolio.
Each row = one cardholder with aggregated metrics across their active tenure.
**18 raw features** covering balances, purchases, cash advances, payment behaviour, and relationship length.

> ⚠️ *This data is proprietary to ABC Debt Relief and is classified as Confidential.*

In [ ]:
df_raw = pd.read_csv('Customer_Data (1).csv')
print(f"Dataset shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head()

In [ ]:
# Feature data dictionary
feat_dict = {
    'CUST_ID':                          'Credit card holder ID (drop before modelling)',
    'BALANCE':                          'Monthly average balance (based on daily averages)',
    'BALANCE_FREQUENCY':                'Frequency balance is updated (0 = rarely, 1 = always)',
    'PURCHASES':                        'Total purchase amount during tenure',
    'ONEOFF_PURCHASES':                 'Maximum one-off purchase amount',
    'INSTALLMENTS_PURCHASES':           'Total installment purchase amount',
    'CASH_ADVANCE':                     'Total cash advanced by cardholder',
    'PURCHASES_FREQUENCY':              'Frequency of purchases (0–1)',
    'ONEOFF_PURCHASES_FREQUENCY':       'Frequency of one-off purchases (0–1)',
    'PURCHASES_INSTALLMENTS_FREQUENCY': 'Frequency of installment purchases (0–1)',
    'CASH_ADVANCE_FREQUENCY':           'Frequency of cash advances (0–1)',
    'CASH_ADVANCE_TRX':                 'Number of cash advance transactions',
    'PURCHASES_TRX':                    'Number of purchase transactions',
    'CREDIT_LIMIT':                     'Credit limit on the card ($)',
    'PAYMENTS':                         'Total amount paid by the customer',
    'MINIMUM_PAYMENTS':                 'Minimum payments due',
    'PRC_FULL_PAYMENT':                 'Percentage of months with full payment (0–1)',
    'TENURE':                           'Duration of credit card relationship (months)',
}
pd.DataFrame(feat_dict.items(), columns=['Feature', 'Description'])

In [ ]:
# Summary statistics
df_raw.describe().round(2).T.style.background_gradient(cmap='Blues', subset=['mean','std','50%'])

In [ ]:
# Data types, null counts, unique counts
info_df = pd.DataFrame({
    'dtype':      df_raw.dtypes,
    'non_null':   df_raw.notnull().sum(),
    'null_count': df_raw.isnull().sum(),
    'null_%':     (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'unique':     df_raw.nunique(),
})
display(info_df)
print(f"\nTotal memory: {df_raw.memory_usage(deep=True).sum()/1024:.1f} KB")

---
<a id='missing'></a>
## 3️⃣ Missing Value Treatment

Two features have missing values:
- **`CREDIT_LIMIT`** — 1 record (0.01%) → **DROP** (single anomalous record with no usable credit data)
- **`MINIMUM_PAYMENTS`** — 313 records (3.5%) → **IMPUTE** with tenure-stratified mean
  *(Rationale: payment obligations differ by tenure; using global mean would bias younger accounts)*

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = missing / len(df_raw) * 100

fig, ax = plt.subplots(figsize=(7, 3.5))
cols_miss = missing[missing > 0].index.tolist()
vals_miss = missing[missing > 0].values
bars = ax.bar(cols_miss, vals_miss,
              color=[PURPLE, DARK_PUR], edgecolor='white', width=0.45, alpha=0.9)
for bar, val, pct in zip(bars, vals_miss, missing_pct[missing > 0].values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            f'{val} ({pct:.2f}%)', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_ylabel('Missing Count')
ax.set_title('Missing Values by Feature', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
df = df_raw.copy()

# ── Step 1: Drop single null CREDIT_LIMIT row ─────────────────────────────────
null_cl = df[df['CREDIT_LIMIT'].isnull()]
print("Null CREDIT_LIMIT record:")
display(null_cl[['CUST_ID','BALANCE','CREDIT_LIMIT','PAYMENTS','TENURE']])
df = df.dropna(subset=['CREDIT_LIMIT']).reset_index(drop=True)
print(f"\nDropped {len(null_cl)} row — remaining: {len(df):,}")

In [ ]:
# ── Step 2: Inspect MINIMUM_PAYMENTS nulls before imputing ────────────────────
null_mp = df[df['MINIMUM_PAYMENTS'].isnull()]
print(f"MINIMUM_PAYMENTS nulls: {len(null_mp)}")
print(f"  Of these, PAYMENTS == 0: {(null_mp['PAYMENTS'] == 0).sum()}")
print(f"  Of these, PAYMENTS > 0 : {(null_mp['PAYMENTS'] > 0).sum()}")
print()
print("Null MINIMUM_PAYMENTS — payment stats:")
display(null_mp[['PAYMENTS','BALANCE','TENURE']].describe().round(2))

In [ ]:
# ── Step 3: Impute MINIMUM_PAYMENTS with tenure-stratified mean ───────────────
df['MINIMUM_PAYMENTS'] = df.groupby('TENURE')['MINIMUM_PAYMENTS'].transform(
    lambda x: x.fillna(x.mean())
)
# Fallback for any remaining NaN (tenure groups that are all-null)
df['MINIMUM_PAYMENTS'] = df['MINIMUM_PAYMENTS'].fillna(df['MINIMUM_PAYMENTS'].mean())

print(f"Missing values remaining: {df.isnull().sum().sum()}")
print("✅ All missing values handled")

# ── Step 4: Drop non-informative CUST_ID column ───────────────────────────────
df = df.drop(columns=['CUST_ID'])
print(f"Shape after cleaning: {df.shape[0]:,} rows × {df.shape[1]} features")

---
<a id='eda'></a>
## 4️⃣ Exploratory Data Analysis (EDA)

Understand distributions, outlier magnitude, correlations, and key business
relationships before feature engineering and modelling.

In [ ]:
# All feature distributions
fig, axes = plt.subplots(6, 3, figsize=(16, 19))
axes = axes.flatten()
for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=45, color=PURPLE, alpha=0.75, edgecolor='white')
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].set_xlabel('')
    m = df[col].median()
    axes[i].axvline(m, color=DARK_PUR, linestyle='--', linewidth=1.2,
                    label=f'med={m:.1f}')
    axes[i].legend(fontsize=6)
for j in range(len(df.columns), len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Feature Distributions (Pre-Outlier-Removal)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Box plots — outlier magnitude per feature
fig, axes = plt.subplots(6, 3, figsize=(16, 19))
axes = axes.flatten()
for i, col in enumerate(df.columns):
    bp = axes[i].boxplot(df[col], vert=True, patch_artist=True,
                         boxprops=dict(facecolor=LIGHT_BG, color=PURPLE),
                         medianprops=dict(color=DARK_PUR, linewidth=2.5),
                         flierprops=dict(marker='o', markersize=2, alpha=0.25, color=PURPLE),
                         whiskerprops=dict(color=PURPLE),
                         capprops=dict(color=PURPLE))
    axes[i].set_title(col, fontsize=9, fontweight='bold')
for j in range(len(df.columns), len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Box Plots — Outlier Extent per Feature', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(14, 11))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.4,
            annot_kws={'size': 7}, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# Top correlations
corr_pairs = corr.abs().unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1.0].drop_duplicates()
print("Top 10 correlated feature pairs:")
display(corr_pairs.head(10).reset_index().rename(
    columns={'level_0':'Feature A','level_1':'Feature B',0:'|Correlation|}))

In [ ]:
# Key business scatter plots
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

axes[0,0].scatter(df['CREDIT_LIMIT'], df['BALANCE'], alpha=0.12, s=8, color=PURPLE)
axes[0,0].set_xlabel('Credit Limit ($)'); axes[0,0].set_ylabel('Balance ($)')
axes[0,0].set_title('Balance vs Credit Limit')

axes[0,1].scatter(df['PURCHASES'], df['CASH_ADVANCE'], alpha=0.12, s=8, color=DARK_PUR)
axes[0,1].set_xlabel('Purchases ($)'); axes[0,1].set_ylabel('Cash Advance ($)')
axes[0,1].set_title('Purchases vs Cash Advance\n(Spending vs Emergency Borrowing)')

axes[0,2].hist(df['PRC_FULL_PAYMENT'], bins=22, color='#2ecc71', alpha=0.8, edgecolor='white')
axes[0,2].set_xlabel('% Full Payment (0–1)')
axes[0,2].set_title(f'Full Payment Rate\n(Mean {df["PRC_FULL_PAYMENT"].mean():.2f}, '
                    f'Median {df["PRC_FULL_PAYMENT"].median():.2f})')

tc = df['TENURE'].value_counts().sort_index()
axes[1,0].bar(tc.index, tc.values, color=PURPLE, alpha=0.85)
axes[1,0].set_xlabel('Tenure (months)'); axes[1,0].set_ylabel('Count')
axes[1,0].set_title('Tenure Distribution')

axes[1,1].scatter(df['BALANCE'], df['PAYMENTS'], alpha=0.12, s=8, color='#ff9800')
axes[1,1].set_xlabel('Balance ($)'); axes[1,1].set_ylabel('Payments ($)')
axes[1,1].set_title('Payments vs Balance')

axes[1,2].scatter(df['PURCHASES_FREQUENCY'], df['CASH_ADVANCE_FREQUENCY'],
                  alpha=0.12, s=8, color='#d32f2f')
axes[1,2].set_xlabel('Purchase Frequency (0–1)')
axes[1,2].set_ylabel('Cash Advance Frequency (0–1)')
axes[1,2].set_title('Purchase vs Cash Advance Frequency')

plt.suptitle('Key Business Relationships', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Frequency features (all on 0–1 scale)
freq_cols = ['BALANCE_FREQUENCY','PURCHASES_FREQUENCY','ONEOFF_PURCHASES_FREQUENCY',
             'PURCHASES_INSTALLMENTS_FREQUENCY','CASH_ADVANCE_FREQUENCY']
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, freq_cols):
    ax.hist(df[col], bins=22, color=PURPLE, alpha=0.8, edgecolor='white')
    ax.axvline(df[col].mean(), color=DARK_PUR, linestyle='--', linewidth=1.5)
    ax.set_title(col.replace('_FREQUENCY','').replace('_',' '), fontsize=8, fontweight='bold')
    ax.set_xlabel('Frequency (0–1)')
plt.suptitle('All Frequency Features (0 = never, 1 = always)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Pairplot on top 6 most informative features (sample for speed)
sample_eda = df[['BALANCE','PURCHASES','CASH_ADVANCE','CREDIT_LIMIT',
                 'PRC_FULL_PAYMENT','TENURE']].sample(n=1500, random_state=42)
g = sns.pairplot(sample_eda, diag_kind='hist', plot_kws=dict(alpha=0.25, s=8, color=PURPLE),
                 diag_kws=dict(color=PURPLE, alpha=0.75))
g.fig.suptitle('Pairplot — Top 6 Features (n=1,500 sample)', y=1.02, fontsize=12, fontweight='bold')
plt.show()

---
<a id='features'></a>
## 5️⃣ Feature Engineering

7 business-aligned derived metrics computed from the 18 raw features.
These capture the **credit stress**, **payment discipline**, **engagement intensity**,
and **relationship depth** dimensions that drive customer value and risk.

| # | Metric | Formula | High = | Low = |
|---|--------|---------|--------|-------|
| 1 | Utilization Rate | Balance / Credit Limit | Stress | Healthy |
| 2 | Payment Quality | % Full Payments | Creditworthy | Default risk |
| 3 | Engagement Score | Avg(Purchase Freq, CashAdv Freq) | Active | Churn risk |
| 4 | Payment Consistency | Payments / Min Payments (cap 20x) | Aggressive paydown | Min-payer |
| 5 | Debt to Limit | Balance / Credit Limit | Stress (>70%) | Healthy (<20%) |
| 6 | Tenure Score | Tenure / 12 | Long-term | New account |
| 7 | Transaction Volume | Purchase TRX + CashAdv TRX | High activity | Inactive |

In [ ]:
df_eng = df.copy()

# 1 — Utilization Rate
df_eng['Utilization_Rate']    = df_eng['BALANCE'] / (df_eng['CREDIT_LIMIT'] + 1e-9)

# 2 — Payment Quality (direct alias for full payment %)
df_eng['Payment_Quality']     = df_eng['PRC_FULL_PAYMENT']

# 3 — Engagement Score
df_eng['Engagement_Score']    = (df_eng['PURCHASES_FREQUENCY'] +
                                  df_eng['CASH_ADVANCE_FREQUENCY']) / 2

# 4 — Payment Consistency (cap at 20x to avoid extreme outliers)
df_eng['Payment_Consistency'] = np.clip(
    df_eng['PAYMENTS'] / (df_eng['MINIMUM_PAYMENTS'] + 1e-9), 0, 20)

# 5 — Debt to Limit (same as Utilization, useful for interpretability)
df_eng['Debt_to_Limit']       = df_eng['BALANCE'] / (df_eng['CREDIT_LIMIT'] + 1e-9)

# 6 — Tenure Score (normalised 0–1)
df_eng['Tenure_Score']        = df_eng['TENURE'] / 12.0

# 7 — Transaction Volume
df_eng['Transaction_Volume']  = df_eng['PURCHASES_TRX'] + df_eng['CASH_ADVANCE_TRX']

ENG_FEATS = ['Utilization_Rate','Payment_Quality','Engagement_Score',
             'Payment_Consistency','Debt_to_Limit','Tenure_Score','Transaction_Volume']

print("Engineered feature summary statistics:")
df_eng[ENG_FEATS].describe().T.round(3).style.background_gradient(cmap='Blues', subset=['mean'])

In [ ]:
# Distribution of engineered features
eng_colors = ['#667eea','#764ba2','#d32f2f','#2ecc71','#ff9800','#2196f3','#9c27b0']
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, (feat, col) in enumerate(zip(ENG_FEATS, eng_colors)):
    axes[i].hist(df_eng[feat], bins=40, color=col, alpha=0.8, edgecolor='white')
    axes[i].set_title(feat.replace('_',' '), fontsize=9, fontweight='bold')
    axes[i].axvline(df_eng[feat].mean(), color='black', linestyle='--', linewidth=1.2)
    axes[i].set_xlabel('Value')
axes[-1].set_visible(False)
plt.suptitle('Engineered Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
<a id='outliers'></a>
## 6️⃣ Outlier Detection & Removal

Credit card data is highly skewed with extreme tails.
**Isolation Forest** (contamination = 5%) is used because:
- Tree-based and assumption-free (no normality assumption needed)
- Scales well to high-dimensional data
- Well-suited to skewed financial distributions
- Does not require pre-specification of distance thresholds

In [ ]:
df_model = df_eng.copy()
n_before = len(df_model)

iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1)
iso.fit(df_model)
scores  = iso.decision_function(df_model)
anomaly = iso.predict(df_model)          # +1 = inlier, -1 = outlier

df_model['_score']     = scores
df_model['_is_outlier']= (anomaly == -1)
n_out = (anomaly == -1).sum()

print(f"Records before removal : {n_before:,}")
print(f"Outliers detected (5%) : {n_out:,}  ({n_out/n_before*100:.1f}%)")
print(f"Records after removal  : {n_before - n_out:,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Score distribution
axes[0].hist(scores[anomaly == 1],  bins=45, color=PURPLE,   alpha=0.75, label='Inlier',
             edgecolor='white')
axes[0].hist(scores[anomaly == -1], bins=45, color='#d32f2f', alpha=0.75, label='Outlier',
             edgecolor='white')
axes[0].axvline(0, color='black', linestyle='--', linewidth=1.8, label='Decision boundary')
axes[0].set_xlabel('Anomaly Score'); axes[0].set_ylabel('Count')
axes[0].set_title('Isolation Forest Score Distribution', fontweight='bold')
axes[0].legend()

# Mean feature values — inliers vs outliers
num_cols = ['BALANCE','PURCHASES','CASH_ADVANCE','CREDIT_LIMIT','PAYMENTS']
ou_means = df_model[df_model['_is_outlier']][num_cols].mean()
in_means = df_model[~df_model['_is_outlier']][num_cols].mean()
x = np.arange(len(num_cols)); w = 0.35
axes[1].bar(x - w/2, in_means.values/1000, w, label='Inliers',  color=PURPLE,    alpha=0.85)
axes[1].bar(x + w/2, ou_means.values/1000, w, label='Outliers', color='#d32f2f', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels([c[:10] for c in num_cols], rotation=30, ha='right')
axes[1].set_ylabel('Mean Value ($K)')
axes[1].set_title('Inlier vs Outlier — Mean Feature Values', fontweight='bold')
axes[1].legend()

plt.suptitle('Isolation Forest — Outlier Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Remove outliers and finalise clean dataset
df_clean = (df_model[~df_model['_is_outlier']]
            .drop(columns=['_score','_is_outlier'])
            .reset_index(drop=True))
print(f"✅ Clean dataset ready: {len(df_clean):,} rows × {df_clean.shape[1]} features")

---
<a id='preprocessing'></a>
## 7️⃣ Data Preprocessing & Scaling

K-Means uses Euclidean distance — raw dollar features (BALANCE in thousands, TENURE in 1–12)
will dominate unless normalised. **StandardScaler** (z-score) ensures equal feature weighting.

In [ ]:
CLUSTER_FEATS = [c for c in df_clean.columns]   # all 25 features (18 raw + 7 engineered)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[CLUSTER_FEATS])
X_df     = pd.DataFrame(X_scaled, columns=CLUSTER_FEATS)

print(f"Scaling complete: {X_scaled.shape[0]:,} samples × {X_scaled.shape[1]} features")
print(f"Post-scale mean (≈ 0): {X_scaled.mean():.6f}")
print(f"Post-scale std  (≈ 1): {X_scaled.std():.6f}")

In [ ]:
# Before vs after scaling
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
check_cols  = ['BALANCE','PURCHASES','CASH_ADVANCE','CREDIT_LIMIT']

axes[0].boxplot([df_clean[c] for c in check_cols],
                labels=check_cols, patch_artist=True,
                boxprops=dict(facecolor=LIGHT_BG, color=PURPLE),
                medianprops=dict(color=DARK_PUR, linewidth=2))
axes[0].set_title('Before Scaling (raw $ values)', fontweight='bold')
axes[0].set_ylabel('Value ($)'); axes[0].tick_params(axis='x', rotation=20)

axes[1].boxplot([X_df[c] for c in check_cols],
                labels=check_cols, patch_artist=True,
                boxprops=dict(facecolor=LIGHT_BG, color=PURPLE),
                medianprops=dict(color=DARK_PUR, linewidth=2))
axes[1].set_title('After Scaling (z-scores)', fontweight='bold')
axes[1].set_ylabel('z-score'); axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('Effect of StandardScaler on Key Features', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
<a id='elbow'></a>
## 8️⃣ Cluster Selection — Elbow Method (Inertia)

**Within-cluster sum of squared distances (WCSS / Inertia)** measures how tight clusters are.
Fit K-Means for k = 1…15 and identify the **elbow** — the point where marginal improvement falls sharply.
Below the elbow, adding more clusters yields diminishing returns vs. operational complexity.

In [ ]:
KM_KWARGS = {'init': 'k-means++', 'n_init': 15, 'max_iter': 500, 'random_state': 42}

inertias = []
K_RANGE  = range(1, 16)

for k in K_RANGE:
    km = KMeans(n_clusters=k, **KM_KWARGS)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    print(f"  k={k:2d}  inertia={km.inertia_:,.0f}")

In [ ]:
# Inertia table with % improvement
rows = []
for i, (k, inert) in enumerate(zip(K_RANGE, inertias)):
    pct = abs(inertias[i-1]-inert)/inertias[i-1]*100 if i > 0 else np.nan
    rows.append({'k': k, 'Inertia': round(inert,1), '% Drop from k-1': round(pct,1) if not np.isnan(pct) else '—'})
pd.DataFrame(rows).set_index('k')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow line plot
axes[0].fill_between(list(K_RANGE), inertias, alpha=0.12, color=PURPLE)
axes[0].plot(list(K_RANGE), inertias, 'o-', color=PURPLE, linewidth=2.5,
             markersize=8, markerfacecolor=DARK_PUR, markeredgecolor='white', markeredgewidth=1.5)
axes[0].scatter([5], [inertias[4]], s=250, zorder=6, color='#ffc107',
                edgecolors='white', linewidth=2)
axes[0].annotate('k = 5\n← ELBOW', xy=(5, inertias[4]),
                 xytext=(7.5, inertias[4] + (inertias[0]-inertias[-1])*0.1),
                 fontsize=10, fontweight='bold', color='#c17f00',
                 arrowprops=dict(arrowstyle='->', color='#c17f00', lw=1.8))
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia (WCSS)', fontsize=12)
axes[0].set_title('Elbow Method — Inertia Curve', fontweight='bold')
axes[0].set_xticks(list(K_RANGE)); axes[0].grid(axis='y', alpha=0.3)

# % Improvement bar chart
pct_impr = [abs(inertias[i]-inertias[i-1])/inertias[i-1]*100 for i in range(1, len(inertias))]
ks2 = list(range(2, 16))
bar_cols = ['#ffc107' if k == 5 else PURPLE for k in ks2]
bars = axes[1].bar(ks2, pct_impr, color=bar_cols, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, pct_impr):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.15,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=8)
axes[1].set_xlabel('k'); axes[1].set_ylabel('% Inertia Reduction vs k-1')
axes[1].set_title('Marginal Gain from Adding One Cluster', fontweight='bold')
axes[1].set_xticks(ks2)

plt.suptitle('Elbow Method Analysis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

if KNEED:
    kl = KneeLocator(list(K_RANGE), inertias, curve='convex', direction='decreasing')
    print(f"KneeLocator automated detection: k = {kl.knee}")

---
<a id='silhouette'></a>
## 9️⃣ Cluster Selection — Silhouette Score & Validation Metrics

Three independent validation metrics computed for k = 2…13:
- **Silhouette Score** (↑ higher = better, range −1 to +1) — separation vs cohesion
- **Davies-Bouldin Index** (↓ lower = better) — average cluster scatter ratio
- **Calinski-Harabasz Score** (↑ higher = better) — ratio of between-cluster to within-cluster variance

In [ ]:
sil_scores, db_scores, ch_scores = [], [], []
K_RANGE2 = range(2, 14)

for k in K_RANGE2:
    km = KMeans(n_clusters=k, **KM_KWARGS)
    labels = km.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))
    ch_scores.append(calinski_harabasz_score(X_scaled, labels))
    print(f"  k={k:2d}  sil={sil_scores[-1]:.4f}  DB={db_scores[-1]:.4f}  CH={ch_scores[-1]:.0f}")

In [ ]:
metrics_df = pd.DataFrame({
    'k':               list(K_RANGE2),
    'Silhouette ↑':    [round(s, 4) for s in sil_scores],
    'Davies-Bouldin ↓':[round(d, 4) for d in db_scores],
    'Calinski-H ↑':    [round(c, 1) for c in ch_scores],
}).set_index('k')

metrics_df.style     .highlight_max(subset=['Silhouette ↑'],    color='#b6f5b6')     .highlight_min(subset=['Davies-Bouldin ↓'], color='#b6f5b6')     .highlight_max(subset=['Calinski-H ↑'],    color='#b6f5b6')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
k_list = list(K_RANGE2)

def _annotate_k5(ax, k5_val, offset_y=0):
    ax.scatter([5], [k5_val], s=240, zorder=6, color='#ffc107',
               edgecolors='white', linewidth=2)
    ax.annotate('k=5', xy=(5, k5_val), xytext=(7, k5_val + offset_y),
                fontsize=9, fontweight='bold', color='#c17f00',
                arrowprops=dict(arrowstyle='->', color='#c17f00'))

# Silhouette
axes[0].fill_between(k_list, sil_scores, alpha=0.15, color='#2ecc71')
axes[0].plot(k_list, sil_scores, 'o-', color='#2ecc71', linewidth=2.5, markersize=7)
_annotate_k5(axes[0], sil_scores[3], offset_y=0.003)
axes[0].set_xlabel('k'); axes[0].set_ylabel('Score')
axes[0].set_title('Silhouette Score (↑ better)', fontweight='bold')
axes[0].set_xticks(k_list)

# Davies-Bouldin
axes[1].plot(k_list, db_scores, 'o-', color='#d32f2f', linewidth=2.5, markersize=7)
_annotate_k5(axes[1], db_scores[3], offset_y=0.05)
axes[1].set_xlabel('k'); axes[1].set_title('Davies-Bouldin Index (↓ better)', fontweight='bold')
axes[1].set_xticks(k_list)

# Calinski-Harabasz
axes[2].plot(k_list, ch_scores, 'o-', color='#2196f3', linewidth=2.5, markersize=7)
_annotate_k5(axes[2], ch_scores[3], offset_y=50)
axes[2].set_xlabel('k'); axes[2].set_title('Calinski-Harabasz Score (↑ better)', fontweight='bold')
axes[2].set_xticks(k_list)

plt.suptitle('Three Validation Metrics — All Indicate k=5 as Optimal Practical Choice',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Combined decision chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].fill_between(list(K_RANGE)[1:], inertias[1:], alpha=0.12, color=PURPLE)
axes[0].plot(list(K_RANGE)[1:], inertias[1:], 'o-', color=PURPLE, linewidth=2.5, markersize=7)
axes[0].scatter([5], [inertias[4]], s=220, color='#ffc107', zorder=5, edgecolors='white')
axes[0].annotate('k=5 elbow', xy=(5, inertias[4]),
                 xytext=(8, inertias[4] + inertias[1]*0.06),
                 fontsize=9, fontweight='bold', color='#c17f00',
                 arrowprops=dict(arrowstyle='->', color='#c17f00'))
axes[0].set_title('Elbow: Clear inflection at k=5', fontweight='bold')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia'); axes[0].set_xticks(list(K_RANGE)[1:])

axes[1].fill_between(k_list, sil_scores, alpha=0.15, color='#2ecc71')
axes[1].plot(k_list, sil_scores, 'o-', color='#2ecc71', linewidth=2.5, markersize=7)
axes[1].scatter([5], [sil_scores[3]], s=220, color='#ffc107', zorder=5, edgecolors='white')
sil_peak = max(sil_scores); k_peak = k_list[sil_scores.index(sil_peak)]
axes[1].annotate(f'Peak k={k_peak} ({sil_peak:.3f})',
                 xy=(k_peak, sil_peak), xytext=(k_peak-3, sil_peak-0.01),
                 fontsize=8, color='#666',
                 arrowprops=dict(arrowstyle='->', color='#aaa'))
axes[1].annotate(f'k=5: {sil_scores[3]:.3f}\n(+{(sil_peak-sil_scores[3])/sil_scores[3]*100:.0f}% vs peak)',
                 xy=(5, sil_scores[3]), xytext=(8, sil_scores[3]-0.015),
                 fontsize=9, fontweight='bold', color='#c17f00',
                 arrowprops=dict(arrowstyle='->', color='#c17f00'))
axes[1].set_title('Silhouette: k=5 good; marginal peak gain not worth complexity', fontweight='bold')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score'); axes[1].set_xticks(k_list)

plt.suptitle('✅ k=5 Decision: Elbow + Silhouette Consensus', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"\nk=5 Silhouette:  {sil_scores[3]:.4f}")
print(f"k={k_peak} Silhouette: {sil_peak:.4f}  (peak, +{(sil_peak-sil_scores[3])/sil_scores[3]*100:.1f}%)")
print("\n💡 A +3% silhouette gain at k>7 does NOT justify 60% added operational complexity.")
print("   Elbow method confirms clear inflection at k=5. → k=5 SELECTED.")

---
<a id='comparison'></a>
## 1️⃣0️⃣ Model Comparison: K-Means vs Hierarchical vs DBSCAN vs GMM

Four clustering algorithms evaluated. Best algorithm selected based on
silhouette score, interpretability, and operational scalability at production scale.

In [ ]:
results = {}

# ── 1. K-Means ────────────────────────────────────────────────────────────────
km5       = KMeans(n_clusters=5, **KM_KWARGS)
km5_lbl   = km5.fit_predict(X_scaled)
results['K-Means (k=5)'] = dict(
    labels=km5_lbl,
    sil=silhouette_score(X_scaled, km5_lbl),
    db=davies_bouldin_score(X_scaled, km5_lbl),
    ch=calinski_harabasz_score(X_scaled, km5_lbl),
    k=5
)

# ── 2. Agglomerative (Ward linkage) ───────────────────────────────────────────
agg       = AgglomerativeClustering(n_clusters=5, metric='euclidean', linkage='ward')
agg_lbl   = agg.fit_predict(X_scaled)
results['Hierarchical (k=5)'] = dict(
    labels=agg_lbl,
    sil=silhouette_score(X_scaled, agg_lbl),
    db=davies_bouldin_score(X_scaled, agg_lbl),
    ch=calinski_harabasz_score(X_scaled, agg_lbl),
    k=5
)

# ── 3. DBSCAN (eps via 5-NN distance percentile) ──────────────────────────────
nn = NearestNeighbors(n_neighbors=5); nn.fit(X_scaled)
d5, _ = nn.kneighbors(X_scaled)
eps_v  = np.percentile(d5[:, 4], 92)
dbs    = DBSCAN(eps=eps_v, min_samples=10)
dbs_lbl = dbs.fit_predict(X_scaled)
n_dbs   = len(set(dbs_lbl)) - (1 if -1 in dbs_lbl else 0)
noise_n = (dbs_lbl == -1).sum()
valid   = dbs_lbl[dbs_lbl != -1]
X_valid = X_scaled[dbs_lbl != -1]
sil_dbs = silhouette_score(X_valid, valid) if n_dbs >= 2 else np.nan
results['DBSCAN'] = dict(
    labels=dbs_lbl,
    sil=sil_dbs,
    db=davies_bouldin_score(X_valid, valid) if n_dbs >= 2 else np.nan,
    ch=calinski_harabasz_score(X_valid, valid) if n_dbs >= 2 else np.nan,
    k=n_dbs
)
print(f"DBSCAN: eps={eps_v:.3f}, clusters found={n_dbs}, noise points={noise_n}")

# ── 4. Gaussian Mixture Model ─────────────────────────────────────────────────
gmm5    = GaussianMixture(n_components=5, random_state=42, n_init=5)
gmm_lbl = gmm5.fit_predict(X_scaled)
results['GMM (k=5)'] = dict(
    labels=gmm_lbl,
    sil=silhouette_score(X_scaled, gmm_lbl),
    db=davies_bouldin_score(X_scaled, gmm_lbl),
    ch=calinski_harabasz_score(X_scaled, gmm_lbl),
    k=5
)

print("\nAll models trained.")

In [ ]:
# Comparison table
comp_rows = []
for name, res in results.items():
    comp_rows.append({
        'Algorithm':       name,
        'Clusters':        res['k'],
        'Silhouette ↑':    round(res['sil'], 4) if not np.isnan(res['sil']) else 'N/A',
        'Davies-Bouldin ↓':round(res['db'],  4) if not np.isnan(res['db'])  else 'N/A',
        'Calinski-H ↑':    round(res['ch'],  1) if not np.isnan(res['ch'])  else 'N/A',
    })

comp_df = pd.DataFrame(comp_rows).set_index('Algorithm')
display(comp_df)
print("\n✅ K-Means wins on: silhouette score + full interpretability + production scalability")

In [ ]:
# PCA 2D scatter for all four models
X_pca2 = PCA(n_components=2, random_state=42).fit_transform(X_scaled)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()
for ax, (name, res) in zip(axes, results.items()):
    sc = ax.scatter(X_pca2[:,0], X_pca2[:,1], c=res['labels'],
                    cmap='tab10', s=7, alpha=0.45, linewidths=0)
    sil_str = f"{res['sil']:.3f}" if not np.isnan(res['sil']) else 'N/A'
    ax.set_title(f"{name}  |  Silhouette = {sil_str}", fontweight='bold', fontsize=10)
    ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
plt.suptitle('Cluster Comparison — PCA 2D Projection', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Dendrogram — 500-sample subset for speed
np.random.seed(42)
idx   = np.random.choice(len(X_scaled), size=min(500, len(X_scaled)), replace=False)
lmat  = linkage(X_scaled[idx], method='ward')
cut_h = np.sort(lmat[:, 2])[-4]   # cut at k=5

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(lmat, ax=ax, truncate_mode='lastp', p=30,
           leaf_rotation=90, leaf_font_size=8, color_threshold=cut_h)
ax.axhline(y=cut_h, color='#ffc107', linestyle='--', linewidth=1.8, label='Cut → 5 clusters')
ax.set_title("Hierarchical Clustering Dendrogram (500-sample, Ward linkage)", fontweight='bold')
ax.set_xlabel('Sample'); ax.set_ylabel("Ward's Distance")
ax.legend(); plt.tight_layout(); plt.show()

---
<a id='finalmodel'></a>
## 1️⃣1️⃣ Final Model — K-Means k=5

Fit the production model and assign interpretable segment names
based on centroid analysis of key business metrics.

In [ ]:
# Fit final model
final_km             = KMeans(n_clusters=5, **KM_KWARGS)
df_clean['Cluster']  = final_km.fit_predict(X_scaled)

print("Cluster sizes:")
print(df_clean['Cluster'].value_counts().sort_index().to_string())
print(f"\nFinal Silhouette Score: {silhouette_score(X_scaled, df_clean['Cluster']):.4f}")

In [ ]:
# Centroid analysis — back-transform to original scale
centroid_df = pd.DataFrame(
    scaler.inverse_transform(final_km.cluster_centers_),
    columns=CLUSTER_FEATS
)
key_c = ['BALANCE','PURCHASES','CASH_ADVANCE','CREDIT_LIMIT','PAYMENTS',
         'PRC_FULL_PAYMENT','Utilization_Rate','Engagement_Score','TENURE']
print("Cluster centroids (original scale):")
centroid_df[key_c].round(3).style     .background_gradient(cmap='Reds',   subset=['Utilization_Rate'])     .background_gradient(cmap='Greens', subset=['PRC_FULL_PAYMENT','PAYMENTS'])     .format('{:.2f}')

In [ ]:
# ── Map raw cluster IDs → segment names ──────────────────────────────────────
c = centroid_df[key_c].copy()
seg_map = {}

# Distressed: highest utilization
seg_map[c['Utilization_Rate'].idxmax()] = 'Distressed Revolvers'
# Prime: highest full payment rate
seg_map[c['PRC_FULL_PAYMENT'].idxmax()] = 'Prime Customers'
# Engaged: highest engagement among remaining
rem = [i for i in range(5) if i not in seg_map]
seg_map[c.loc[rem, 'Engagement_Score'].idxmax()] = 'Engaged Optimizers'
rem = [i for i in range(5) if i not in seg_map]
# High Risk/New: shortest tenure
seg_map[c.loc[rem, 'TENURE'].idxmin()] = 'High Risk / New'
rem = [i for i in range(5) if i not in seg_map]
seg_map[rem[0]] = 'Low Cap / Disengaged'

print("Cluster → Segment mapping:")
for k, v in sorted(seg_map.items()):
    n = (df_clean['Cluster'] == k).sum()
    print(f"  Cluster {k} → {v}  (n={n:,}, {n/len(df_clean)*100:.1f}%)")

df_clean['Segment'] = df_clean['Cluster'].map(seg_map)

In [ ]:
# Segment size charts
seg_counts = df_clean['Segment'].value_counts()
seg_ord    = [s for s in SEG_ORDER if s in seg_counts.index]
seg_counts = seg_counts.reindex(seg_ord)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar
bars = axes[0].bar(range(len(seg_counts)), seg_counts.values,
                   color=[SEG_COLS[s] for s in seg_counts.index], alpha=0.88, edgecolor='white')
for bar, val in zip(bars, seg_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+8,
                 f'{val:,}\n({val/len(df_clean)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_xticks(range(len(seg_counts)))
axes[0].set_xticklabels([s.replace(' /','/\n').replace(' ','
',1) if len(s)>12 else s
                          for s in seg_counts.index], fontsize=8)
axes[0].set_ylabel('Customer Count')
axes[0].set_title('Segment Sizes', fontweight='bold')

# Pie
wedges, texts, autotexts = axes[1].pie(
    seg_counts.values, labels=seg_counts.index,
    colors=[SEG_COLS[s] for s in seg_counts.index],
    autopct='%1.1f%%', startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2))
for t in autotexts:
    t.set_fontsize(9); t.set_fontweight('bold')
axes[1].set_title('Portfolio Distribution', fontweight='bold')

plt.suptitle(f'K-Means k=5 Segment Distribution  (n={len(df_clean):,})',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Segment profile heatmap
seg_metrics = ['BALANCE','Utilization_Rate','Payment_Quality','Engagement_Score',
               'PURCHASES','CASH_ADVANCE','PRC_FULL_PAYMENT','Transaction_Volume','TENURE']
seg_means   = df_clean.groupby('Segment')[seg_metrics].mean()
seg_norm    = (seg_means - seg_means.min()) / (seg_means.max() - seg_means.min())

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(seg_norm.T, annot=seg_means.T.round(2), fmt='g', cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Normalised (0–1)'})
ax.set_title('Segment Profile Heatmap — Normalised Feature Means', fontsize=13, fontweight='bold')
ax.set_ylabel('')
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.tight_layout(); plt.show()

---
<a id='pca'></a>
## 1️⃣2️⃣ PCA Cluster Visualisation

Project 25-dimensional scaled features to 2D and inspect visual cluster separation.

In [ ]:
pca_full = PCA(random_state=42).fit(X_scaled)
exp_var  = pca_full.explained_variance_ratio_ * 100
cumvar   = np.cumsum(exp_var)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree
axes[0].bar(range(1, 16), exp_var[:15], color=PURPLE, alpha=0.85, edgecolor='white', label='Per-component')
axes[0].plot(range(1, 16), cumvar[:15], 'o-', color=DARK_PUR, linewidth=2, label='Cumulative')
axes[0].axhline(80, color='#d32f2f', linestyle='--', linewidth=1.2, label='80% threshold')
n80 = next(i+1 for i, v in enumerate(cumvar) if v >= 80)
axes[0].axvline(n80, color='#d32f2f', linestyle=':', linewidth=1.2)
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('% Variance Explained')
axes[0].set_title(f'Scree Plot (PC1–{n80} explain ≥80%)', fontweight='bold')
axes[0].legend(fontsize=9)

# 2D scatter coloured by segment
X_pca_2 = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
for seg, col in SEG_COLS.items():
    mask = df_clean['Segment'] == seg
    if mask.sum() > 0:
        axes[1].scatter(X_pca_2[mask, 0], X_pca_2[mask, 1],
                        c=col, label=seg, s=12, alpha=0.45, linewidths=0)
axes[1].set_xlabel(f'PC 1 ({exp_var[0]:.1f}%)'); axes[1].set_ylabel(f'PC 2 ({exp_var[1]:.1f}%)')
axes[1].set_title('Segments in PCA 2D Space', fontweight='bold')
axes[1].legend(fontsize=7, markerscale=2)

plt.suptitle('PCA Analysis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
<a id='segments'></a>
## 1️⃣3️⃣ Segment Profiles & Spider / Radar Charts

Deep-dive into each segment. **Spider charts** show normalised scores (0–100) on 4 dimensions:
> **Utilization Rate** | **Payment Quality** | **Engagement** | **Financial Health**

In [ ]:
# Compute 0–100 normalised radar scores across segments
RADAR_METRICS = ['Utilization Rate','Payment Quality','Engagement','Financial Health']

radar_raw = df_clean.groupby('Segment').agg(
    **{'Utilization Rate': ('Utilization_Rate', 'mean'),
       'Payment Quality':  ('Payment_Quality',  'mean'),
       'Engagement':       ('Engagement_Score', 'mean'),
       'Financial Health': ('PRC_FULL_PAYMENT', 'mean')}
)
radar_norm = radar_raw.apply(
    lambda col: (col - col.min()) / (col.max() - col.min()) * 100
)
print("Radar scores (normalised 0–100 across segments):")
display(radar_norm.round(1))

In [ ]:
def radar_chart_mpl(ax, values, metrics, color, title):
    """Draws a spider/radar chart on a matplotlib polar axis."""
    N = len(metrics)
    angles = [n / float(N) * 2 * math.pi for n in range(N)]
    angles += angles[:1]
    vals   = list(values) + [values[0]]

    # Concentric background rings
    ring_cols = ['#f0f4ff', '#dde5fa', '#ccd5f5']
    for level, rc in zip([100, 66.7, 33.3], ring_cols):
        rv = [level] * N + [level]
        ax.fill(angles, rv, color=rc, alpha=1.0, zorder=1)
        ax.plot(angles, rv, color='#ccccdd', linewidth=0.7, zorder=2)

    # Axis spokes
    for a in angles[:-1]:
        ax.plot([a, a], [0, 100], color='#ccccdd', linewidth=0.9, zorder=3)

    # Data polygon
    ax.fill(angles, vals, color=color, alpha=0.25, zorder=4)
    ax.plot(angles, vals, color=color, linewidth=2.5, zorder=5)
    ax.scatter(angles[:-1], vals[:-1], color=color, s=65,
               zorder=6, edgecolors='white', linewidth=1.8)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, fontsize=8.5, fontweight='bold')
    ax.set_ylim(0, 100); ax.set_yticks([33, 66, 100])
    ax.set_yticklabels(['33', '66', '100'], fontsize=6, color='#888')
    ax.set_title(title, fontsize=9.5, fontweight='bold', pad=16)
    ax.spines['polar'].set_visible(False); ax.grid(False)


fig, axes = plt.subplots(1, 5, figsize=(22, 5), subplot_kw=dict(polar=True))
for ax, seg in zip(axes, SEG_ORDER):
    if seg in radar_norm.index:
        radar_chart_mpl(ax, radar_norm.loc[seg, RADAR_METRICS].values,
                        RADAR_METRICS, SEG_COLS[seg],
                        seg.replace('/','/\n').replace(' ','\n', 1))

plt.suptitle('Segment Spider Charts — Normalised Profile (0–100)',
             fontsize=14, fontweight='bold', y=1.04)
plt.tight_layout(); plt.savefig('segment_spider_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Spider chart saved as segment_spider_charts.png")

### 🔴 Segment 1 — Distressed Revolvers

**Profile:** Near-maxed credit utilisation, negligible full-payment rate, avoidance behaviour.
**Risk:** Highest default risk in portfolio (est. 15–20% default rate).
**Action:** Immediate intervention before balances become unrecoverable.

In [ ]:
S1   = df_clean[df_clean['Segment'] == 'Distressed Revolvers']
KC   = ['BALANCE','CREDIT_LIMIT','Utilization_Rate','PRC_FULL_PAYMENT',
        'PAYMENTS','MINIMUM_PAYMENTS','CASH_ADVANCE','Engagement_Score']
print(f"n = {len(S1):,}  ({len(S1)/len(df_clean)*100:.1f}%)")
S1[KC].describe().T[['mean','std','25%','50%','75%']].round(3).style     .background_gradient(cmap='Reds', subset=['mean'])

In [ ]:
c = '#d32f2f'
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(S1['Utilization_Rate'], bins=35, color=c, alpha=0.8, edgecolor='white')
axes[0].axvline(S1['Utilization_Rate'].mean(), color='black', ls='--', lw=1.5,
                label=f"μ={S1['Utilization_Rate'].mean():.2f}")
axes[0].set_xlabel('Utilization Rate'); axes[0].set_title('Utilization Distribution', fontweight='bold')
axes[0].legend()

axes[1].hist(S1['PRC_FULL_PAYMENT'], bins=20, color=c, alpha=0.8, edgecolor='white')
axes[1].set_xlabel('% Full Payment'); axes[1].set_title('Payment Quality', fontweight='bold')

axes[2].scatter(S1['BALANCE'], S1['CREDIT_LIMIT'], alpha=0.15, s=10, color=c)
axes[2].set_xlabel('Balance ($)'); axes[2].set_ylabel('Credit Limit ($)')
axes[2].set_title('Balance vs Credit Limit', fontweight='bold')

plt.suptitle('🔴 Distressed Revolvers — Feature Distributions', fontsize=12, fontweight='bold', color=c)
plt.tight_layout(); plt.show()

### 🟢 Segment 2 — Prime Customers

**Profile:** Near-zero utilisation, exceptional payment discipline (high full-payment rate),
stable long tenure, strategic spending.
**Value:** Highest LTV ($3,000–$5,000) — the portfolio's crown jewel.
**Action:** Grow, retain, cross-sell — churn is structural loss.

In [ ]:
S2 = df_clean[df_clean['Segment'] == 'Prime Customers']
print(f"n = {len(S2):,}  ({len(S2)/len(df_clean)*100:.1f}%)")
S2[KC].describe().T[['mean','std','25%','50%','75%']].round(3).style     .background_gradient(cmap='Greens', subset=['mean'])

In [ ]:
c = '#2ecc71'
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(S2['PRC_FULL_PAYMENT'], bins=22, color=c, alpha=0.8, edgecolor='white')
axes[0].axvline(S2['PRC_FULL_PAYMENT'].mean(), color='black', ls='--', lw=1.5,
                label=f"μ={S2['PRC_FULL_PAYMENT'].mean():.2f}")
axes[0].set_xlabel('% Full Payment'); axes[0].set_title('Full Payment Rate', fontweight='bold')
axes[0].legend()

axes[1].hist(S2['TENURE'], bins=15, color=c, alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Tenure (months)'); axes[1].set_title('Tenure Distribution', fontweight='bold')

axes[2].scatter(S2['CREDIT_LIMIT'], S2['PURCHASES'], alpha=0.2, s=12, color=c)
axes[2].set_xlabel('Credit Limit ($)'); axes[2].set_ylabel('Purchases ($)')
axes[2].set_title('Credit Limit vs Purchases', fontweight='bold')

plt.suptitle('🟢 Prime Customers — Feature Distributions', fontsize=12, fontweight='bold', color=c)
plt.tight_layout(); plt.show()

### 🟠 Segment 3 — Engaged Optimizers

**Profile:** Highest purchase frequency, peak engagement score, deliberate balance management
(not financial stress). Revenue engine of the portfolio.
**Value:** Reliable recurring revenue, low churn risk.
**Action:** Reward programmes, installment products, business owner community.

In [ ]:
S3 = df_clean[df_clean['Segment'] == 'Engaged Optimizers']
print(f"n = {len(S3):,}  ({len(S3)/len(df_clean)*100:.1f}%)")
S3[KC].describe().T[['mean','std','25%','50%','75%']].round(3).style     .background_gradient(cmap='Oranges', subset=['mean'])

In [ ]:
c = '#ff9800'
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(S3['PURCHASES'], bins=35, color=c, alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Purchases ($)'); axes[0].set_title('Purchase Distribution', fontweight='bold')

axes[1].hist(S3['Engagement_Score'], bins=22, color=c, alpha=0.8, edgecolor='white')
axes[1].axvline(S3['Engagement_Score'].mean(), color='black', ls='--', lw=1.5,
                label=f"μ={S3['Engagement_Score'].mean():.3f}")
axes[1].set_xlabel('Engagement Score'); axes[1].set_title('Engagement Score', fontweight='bold')
axes[1].legend()

axes[2].scatter(S3['PURCHASES'], S3['BALANCE'], alpha=0.2, s=12, color=c)
axes[2].set_xlabel('Purchases ($)'); axes[2].set_ylabel('Balance ($)')
axes[2].set_title('Purchases vs Balance', fontweight='bold')

plt.suptitle('🟠 Engaged Optimizers — Feature Distributions', fontsize=12, fontweight='bold', color=c)
plt.tight_layout(); plt.show()

### 🔵 Segment 4 — Low Cap / Disengaged

**Profile:** Lowest spending, lowest engagement, minimal credit utilisation.
**Risk:** Low revenue, high cost-to-serve relative to contribution. Potential portfolio drag.
**Action:** Activate 50%, digitise 30%, exit 20% — based on individual account P&L.

In [ ]:
S4 = df_clean[df_clean['Segment'] == 'Low Cap / Disengaged']
print(f"n = {len(S4):,}  ({len(S4)/len(df_clean)*100:.1f}%)")
S4[KC].describe().T[['mean','std','25%','50%','75%']].round(3).style     .background_gradient(cmap='Blues', subset=['mean'])

In [ ]:
c = '#2196f3'
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(S4['Engagement_Score'], bins=22, color=c, alpha=0.8, edgecolor='white')
axes[0].axvline(S4['Engagement_Score'].mean(), color='black', ls='--', lw=1.5,
                label=f"μ={S4['Engagement_Score'].mean():.3f}")
axes[0].set_xlabel('Engagement Score'); axes[0].set_title('Engagement Score', fontweight='bold')
axes[0].legend()

axes[1].hist(S4['PURCHASES'], bins=30, color=c, alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Purchases ($)'); axes[1].set_title('Purchase Distribution', fontweight='bold')

axes[2].hist(S4['CREDIT_LIMIT'], bins=30, color=c, alpha=0.8, edgecolor='white')
axes[2].set_xlabel('Credit Limit ($)'); axes[2].set_title('Credit Limit Distribution', fontweight='bold')

plt.suptitle('🔵 Low Cap / Disengaged — Feature Distributions', fontsize=12, fontweight='bold', color=c)
plt.tight_layout(); plt.show()

### 🟣 Segment 5 — High Risk / New Customers

**Profile:** Shortest tenure, emerging financial stress, cash advance dependence growing,
payment difficulty beginning to surface.
**Risk:** 50–60% will migrate to Distressed without early intervention.
**Action:** Early-warning scoring, proactive financial check-ins, credit builder programme.
**ROI of prevention:** 10:1 vs curing Distressed later.

In [ ]:
S5 = df_clean[df_clean['Segment'] == 'High Risk / New']
print(f"n = {len(S5):,}  ({len(S5)/len(df_clean)*100:.1f}%)")
S5[KC].describe().T[['mean','std','25%','50%','75%']].round(3).style     .background_gradient(cmap='Purples', subset=['mean'])

In [ ]:
c = '#9c27b0'
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(S5['TENURE'], bins=14, color=c, alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Tenure (months)'); axes[0].set_title('Tenure — New Accounts', fontweight='bold')

axes[1].hist(S5['Utilization_Rate'], bins=28, color=c, alpha=0.8, edgecolor='white')
axes[1].axvline(S5['Utilization_Rate'].mean(), color='black', ls='--', lw=1.5,
                label=f"μ={S5['Utilization_Rate'].mean():.2f}")
axes[1].set_xlabel('Utilization Rate'); axes[1].set_title('Utilization Rate', fontweight='bold')
axes[1].legend()

axes[2].scatter(S5['TENURE'], S5['CASH_ADVANCE'], alpha=0.2, s=12, color=c)
axes[2].set_xlabel('Tenure (months)'); axes[2].set_ylabel('Cash Advance ($)')
axes[2].set_title('Tenure vs Cash Advance', fontweight='bold')

plt.suptitle('🟣 High Risk / New Customers — Feature Distributions', fontsize=12, fontweight='bold', color=c)
plt.tight_layout(); plt.show()

In [ ]:
# All 5 segments compared — key metrics side by side
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
compare_feats = ['BALANCE','Utilization_Rate','PRC_FULL_PAYMENT',
                 'Engagement_Score','PURCHASES','TENURE']
compare_lbls  = ['Balance ($)','Utilization Rate','Full Payment %',
                 'Engagement Score','Purchases ($)','Tenure (months)']

for ax, feat, lbl in zip(axes, compare_feats, compare_lbls):
    segs_ord = [s for s in SEG_ORDER if s in df_clean['Segment'].unique()]
    data_bx  = [df_clean[df_clean['Segment']==s][feat].values for s in segs_ord]
    bp = ax.boxplot(data_bx, patch_artist=True, showfliers=False,
                    medianprops=dict(linewidth=2.5, color='white'))
    for patch, seg in zip(bp['boxes'], segs_ord):
        patch.set_facecolor(SEG_COLS[seg]); patch.set_alpha(0.75)
    ax.set_xticks(range(1, len(segs_ord)+1))
    ax.set_xticklabels([s.split()[0] for s in segs_ord], fontsize=8)
    ax.set_ylabel(lbl, fontsize=9); ax.set_title(lbl, fontweight='bold')

plt.suptitle('Cross-Segment Feature Comparison', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
<a id='portfolio'></a>
## 1️⃣4️⃣ Portfolio Distribution & ROI Analysis

In [ ]:
# Full summary table
summary = []
for seg in SEG_ORDER:
    sub = df_clean[df_clean['Segment']==seg]
    if len(sub) == 0: continue
    summary.append({
        'Segment':         seg,
        'Count':           f"{len(sub):,}",
        'Share':           f"{len(sub)/len(df_clean)*100:.1f}%",
        'Avg Balance':     f"${sub['BALANCE'].mean():,.0f}",
        'Avg Credit Lim':  f"${sub['CREDIT_LIMIT'].mean():,.0f}",
        'Utilization':     f"{sub['Utilization_Rate'].mean()*100:.1f}%",
        'Full Pmt':        f"{sub['PRC_FULL_PAYMENT'].mean()*100:.1f}%",
        'Avg Tenure(mo)':  f"{sub['TENURE'].mean():.1f}",
    })
pd.DataFrame(summary).set_index('Segment').style.set_properties(**{'text-align':'center'})

In [ ]:
# Portfolio pie + investment vs benefit
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

seg_c   = df_clean['Segment'].value_counts().reindex(
    [s for s in SEG_ORDER if s in df_clean['Segment'].unique()])
wedges, texts, autotexts = axes[0].pie(
    seg_c.values, labels=seg_c.index,
    colors=[SEG_COLS[s] for s in seg_c.index],
    autopct='%1.1f%%', startangle=140, pctdistance=0.78,
    wedgeprops=dict(edgecolor='white', linewidth=2))
for t in autotexts: t.set_fontsize(9); t.set_fontweight('bold')
axes[0].set_title('Customer Portfolio Distribution', fontweight='bold')

segs_s = ['Distressed','Prime','Engaged','Low Cap','High Risk']
inv_k  = [330, 405, 295, 155, 295]
ben_lo = [1000, 600, 400, 200, 3000]
ben_hi = [2000, 1000, 600, 700, 5000]
x = np.arange(5); w = 0.35

axes[1].bar(x-w/2, inv_k, w, color=ALL_COLS, alpha=0.9, label='Investment ($K)')
axes[1].bar(x+w/2, ben_hi, w, color=ALL_COLS, alpha=0.35,
            edgecolor=ALL_COLS, linewidth=1.5, label='Benefit Max ($K)')
axes[1].bar(x+w/2, ben_lo, w, color=ALL_COLS, alpha=0.75)
for i, (xi, iv) in enumerate(zip(x, inv_k)):
    axes[1].text(xi-w/2, iv+20, f'${iv}K', ha='center', fontsize=7.5, fontweight='bold')
axes[1].set_xticks(x); axes[1].set_xticklabels(segs_s, fontsize=9)
axes[1].set_ylabel('Amount ($K)')
axes[1].set_title('Investment vs Expected Benefit', fontweight='bold')
axes[1].legend()

plt.suptitle('Portfolio Risk & ROI Overview', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ROI horizontal bar
roi_m = [4.0, 3.2, 3.8, 4.7, 4.0]
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(SEG_ORDER, roi_m,
               color=ALL_COLS, alpha=0.85, edgecolor='white', height=0.55)
ax.axvline(2.5, color='#ffc107', ls='--', lw=1.8, label='Min acceptable ROI (2.5x)')
for bar, roi in zip(bars, roi_m):
    ax.text(roi+0.05, bar.get_y()+bar.get_height()/2,
            f'{roi:.1f}x', va='center', fontweight='bold', fontsize=10)
ax.set_xlabel('Expected ROI Multiple', fontsize=11)
ax.set_title('ROI by Segment Initiative', fontsize=13, fontweight='bold')
ax.legend(); ax.set_xlim(0, 6)
plt.tight_layout(); plt.show()

---
<a id='recommendations'></a>
## 1️⃣5️⃣ Recommendations & 90-Day Implementation Roadmap

> **Prepared for:** ABC Debt Relief Leadership Team
> **Scope:** Portfolio-wide segmentation initiative — 5 identified customer clusters

---

### 🎯 Strategic Framework

ABC Debt Relief's five-segment architecture reveals a clear **two-tier portfolio reality**:
- **Revenue creators** (Prime + Engaged): ~29% of customers, disproportionate revenue contribution
- **Risk carriers** (Distressed + High Risk): ~35% of customers, systemic default and loss exposure

All five initiatives can be run concurrently — Distressed and High Risk must launch in Month 1.

---

### Priority 1 — 🔴 URGENT: Distressed Revolvers (Month 1)

| Phase | Action | Target |
|-------|--------|--------|
| Wk 1–2 | Risk triage: Recoverable (40%) / Declining (40%) / Terminal (20%) | Further sub-segment |
| Wk 3–8 | Hardship programmes, balance transfer at lower rates, credit counselling | Reduce default risk |
| Wk 9+ | Auto-payment enrolment; SMS/email alerts at 3 & 7 days pre-due; credit freeze | Stabilise |

**Investment:** $305–355K/yr &nbsp; **Benefit:** $1–2M loss prevention &nbsp; **ROI: 3–5x**

---

### Priority 2 — 🟣 URGENT: High Risk / New Customers (Month 1)

| Phase | Action | Target |
|-------|--------|--------|
| Wk 1–2 | Early-warning scoring: flag late payments, usage spikes, balance acceleration | Early detection |
| Wk 3–8 | Proactive financial check-ins; hardship conversations; income verification | Prevent deterioration |
| Wk 9+ | Credit builder: auto limit-increases for on-time payments; rate reductions | Reward improvement |

**Investment:** $295K/yr &nbsp; **Benefit:** $3–5M default prevention &nbsp; **ROI: 3–5x**

---

### Priority 3 — 🟢 HIGH: Prime Customer Growth (Month 2)

| Phase | Action | Target |
|-------|--------|--------|
| Wk 1–2 | VIP recognition + dedicated service access (before any cross-sell) | Anchor relationship |
| Wk 3–8 | Premium card tier (2–3% cashback); personal loans at below-market rates | Revenue expansion |
| Wk 9+ | Annual VIP events; $100 referral incentives; personalised credit roadmap | LTV maximisation |

**Investment:** $405K/yr &nbsp; **Benefit:** $600K–$1M revenue growth &nbsp; **ROI: 2.4–6x**

---

### Priority 4 — 🟠 HIGH: Engaged Optimizer Retention (Month 2)

| Phase | Action | Target |
|-------|--------|--------|
| Wk 1–4 | Enhanced rewards: 2× points on first $500/month; partner-merchant bonuses | Spending uplift |
| Wk 5–8 | 0% APR installment plans; balance-transfer campaigns | Balance growth |
| Wk 9+ | Business Owner Circle: community, webinars, personalised spending insights | Loyalty deepening |

**Investment:** $295K/yr &nbsp; **Benefit:** +$1.3M balance growth + improved retention &nbsp; **ROI: 4–5x**

---

### Priority 5 — 🔵 EFFICIENCY: Low Cap Activation (Month 3)

| Phase | Action | Target |
|-------|--------|--------|
| Activate 50% | Win-back: 5% cashback on groceries/gas for 3 months; micro-incentives | Reactivation |
| Digitise 30% | Auto-paydown; digital statements; suppress high-touch servicing | Cost reduction |
| Exit 20% | After 90-day attempt: close/downgrade non-responsive accounts | Portfolio hygiene |

**Investment:** $155K/yr &nbsp; **Benefit:** $200K revenue + $500K cost savings &nbsp; **ROI: 4.7x**

---

### 📊 Portfolio-Level Summary

| Initiative | Priority | Investment | Expected Benefit | ROI |
|-----------|----------|-----------|-----------------|-----|
| Distressed Intervention | URGENT — Month 1 | $305–355K | $1–2M loss prevention | 3–5x |
| High Risk Prevention | URGENT — Month 1 | $295K | $3–5M default prevention | 3–5x |
| Prime Growth | HIGH — Month 2 | $405K | $600K–$1M revenue | 2.4x |
| Engaged Optimization | HIGH — Month 2 | $295K | $400–600K revenue | 2.5x |
| Low Cap Activation | EFF — Month 3 | $155K | $200K + $500K savings | 4.7x |
| **TOTAL PORTFOLIO** | **Months 1–3** | **$1.45–1.50M** | **$2.2–4.7M** | **2.5–3.5x** |

---

### 📈 Monthly KPIs to Track

- Default rate by segment *(target: −20% for Distressed within 6 months)*
- Early-warning detection rate *(target: 70%+)*
- Prime upgrade rate from Engaged *(target: 40%)*
- Engaged balance growth *(target: +15% YoY)*
- Churn: Prime >95%, Engaged >92%
- NPS by segment *(baseline + monthly tracking)*

In [ ]:
# Executive dashboard — dark theme
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.patch.set_facecolor('#0d0228')
DARK_BG = '#110335'

def dark_ax(ax):
    ax.set_facecolor(DARK_BG)
    for spine in ax.spines.values(): spine.set_color('#2a1a55')
    ax.tick_params(colors='white'); ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white'); ax.title.set_color('white')

# 1. Segment sizes bar
ax = axes[0, 0]; dark_ax(ax)
seg_ord2 = [s for s in SEG_ORDER if s in df_clean['Segment'].unique()]
sizes2   = [len(df_clean[df_clean['Segment']==s]) for s in seg_ord2]
bars = ax.bar(range(len(seg_ord2)), sizes2,
              color=[SEG_COLS[s] for s in seg_ord2], alpha=0.9)
for bar, val in zip(bars, sizes2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+8,
            f'{val:,}', ha='center', color='white', fontsize=8, fontweight='bold')
ax.set_xticks(range(len(seg_ord2)))
ax.set_xticklabels([s.split()[0] for s in seg_ord2], color='white', fontsize=8)
ax.set_title('Segment Sizes', fontweight='bold')

# 2. Investment bar
ax = axes[0, 1]; dark_ax(ax)
bars = ax.bar(range(5), inv_k, color=ALL_COLS, alpha=0.9)
for bar, v in zip(bars, inv_k):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            f'${v}K', ha='center', color='white', fontsize=8.5, fontweight='bold')
ax.set_xticks(range(5)); ax.set_xticklabels(segs_s, color='white', fontsize=8.5)
ax.set_title('Investment Required ($K)', fontweight='bold')

# 3. ROI bars
ax = axes[0, 2]; dark_ax(ax)
hbars = ax.barh(range(5), roi_m, color=ALL_COLS[::-1], alpha=0.9, height=0.6)
ax.axvline(2.5, color='#ffc107', ls='--', lw=1.8)
for bar, v in zip(hbars, roi_m):
    ax.text(v+0.06, bar.get_y()+bar.get_height()/2, f'{v:.1f}x',
            va='center', color='white', fontsize=9.5, fontweight='bold')
ax.set_yticks(range(5)); ax.set_yticklabels(segs_s[::-1], color='white', fontsize=8.5)
ax.set_xlim(0, 6); ax.set_title('Expected ROI Multiple', fontweight='bold')

# 4. Pie
ax = axes[1, 0]; ax.set_facecolor(DARK_BG)
wedges2, texts2, autotexts2 = ax.pie(
    sizes2, colors=[SEG_COLS[s] for s in seg_ord2],
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='#0d0228', linewidth=2.5),
    pctdistance=0.75)
for t in texts2+autotexts2: t.set_color('white'); t.set_fontsize(7.5)
ax.set_title('Portfolio Split', color='white', fontweight='bold')

# 5. Radar overlay
ax = axes[1, 1]; ax.set_facecolor(DARK_BG)
cats = RADAR_METRICS; xp = np.arange(len(cats))
for seg in seg_ord2:
    if seg in radar_norm.index:
        vals_r = radar_norm.loc[seg, cats].values
        ax.plot(xp, vals_r, 'o-', color=SEG_COLS[seg], lw=2, ms=6,
                label=seg.split()[0], alpha=0.9)
ax.set_xticks(xp); ax.set_xticklabels(cats, color='white', fontsize=7.5)
ax.tick_params(colors='white'); ax.set_ylim(0, 110)
ax.set_title('Radar Scores by Segment', color='white', fontweight='bold')
for spine in ax.spines.values(): spine.set_color('#2a1a55')
ax.legend(fontsize=7, labelcolor='white', facecolor='#1a0845', edgecolor='#444',
          loc='upper right')

# 6. Priority matrix (investment vs ROI)
ax = axes[1, 2]; dark_ax(ax)
for seg, col, iv, ri in zip(seg_ord2, ALL_COLS, inv_k, roi_m):
    ax.scatter(iv, ri, s=280, color=col, zorder=5, edgecolors='white', lw=1.5)
    ax.annotate(seg.split()[0], (iv, ri),
                textcoords='offset points', xytext=(6,4),
                color='white', fontsize=7.5)
ax.axhline(2.5, color='#ffc107', ls='--', lw=1.5, alpha=0.7)
ax.set_xlabel('Investment ($K)', color='white')
ax.set_ylabel('ROI Multiple', color='white')
ax.set_title('Priority Matrix\n(above yellow = acceptable ROI)', fontweight='bold')

plt.suptitle('ABC Debt Relief — Customer Segmentation & ROI Executive Dashboard',
             fontsize=14, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig('executive_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Executive dashboard saved as executive_dashboard.png")